# 🦷 Dental OPG Cavity Detection — Exploratory Data Analysis

**Author:** Paul Sentongo  
**Model:** YOLOv8  
**Task:** Cavity detection in Orthopantomogram (OPG) X-ray images

---

This notebook provides:
1. Dataset overview and statistics
2. Image quality analysis (resolution, contrast, SNR)
3. Annotation distribution analysis
4. Class balance visualization
5. Sample image visualization with annotations
6. Augmentation preview

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print('✓ Imports successful')
print(f'Working directory: {os.getcwd()}')

## 1. Dataset Overview

In [ ]:
# Dataset paths
RAW_DATA = Path('artifacts/data_ingestion/unzipped')
PROCESSED_DATA = Path('artifacts/data_transformation/dataset')

def find_dataset_root(base_path: Path) -> Path:
    if not base_path.exists():
        return None
    contents = list(base_path.iterdir())
    if len(contents) == 1 and contents[0].is_dir():
        return contents[0]
    return base_path

# Find raw data
data_root = find_dataset_root(RAW_DATA)
print(f'Raw data root: {data_root}')

if data_root and data_root.exists():
    # Count files
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    all_images = [f for f in data_root.rglob('*') if f.suffix.lower() in image_exts]
    all_labels = list(data_root.rglob('*.txt')) + list(data_root.rglob('*.xml'))
    all_json = list(data_root.rglob('*.json'))

    print(f'\n📊 Dataset Statistics:')
    print(f'  Total images: {len(all_images)}')
    print(f'  TXT labels: {len([l for l in all_labels if l.suffix == ".txt"])}')
    print(f'  XML labels: {len([l for l in all_labels if l.suffix == ".xml"])}')
    print(f'  JSON files: {len(all_json)}')
    print(f'  Image formats: {set(f.suffix.lower() for f in all_images)}')
else:
    print('⚠️ Raw data not found. Run: python scripts/download_data.py')

## 2. Image Quality Analysis

In [ ]:
def analyze_image_quality(image_paths: list, sample_size: int = 50) -> pd.DataFrame:
    """Analyze image quality metrics for a sample of OPG images."""
    sampled = np.random.choice(image_paths, min(sample_size, len(image_paths)), replace=False)
    
    records = []
    for img_path in sampled:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
        h, w = img.shape[:2]
        
        # Laplacian variance (sharpness)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        
        # Signal-to-noise ratio approximation
        mean_val = gray.mean()
        std_val = gray.std()
        snr = mean_val / std_val if std_val > 0 else 0
        
        # Contrast (max - min normalized)
        contrast = (gray.max() - gray.min()) / 255.0
        
        records.append({
            'filename': img_path.name,
            'width': w,
            'height': h,
            'aspect_ratio': round(w/h, 3),
            'mean_brightness': round(mean_val, 2),
            'std_brightness': round(std_val, 2),
            'sharpness': round(laplacian_var, 2),
            'contrast': round(contrast, 3),
            'snr': round(snr, 3),
            'megapixels': round(h * w / 1e6, 2),
        })
    
    return pd.DataFrame(records)


if data_root and all_images:
    print('Analyzing image quality...')
    df_quality = analyze_image_quality(all_images)
    print('\n📊 Image Quality Statistics:')
    display(df_quality.describe().round(2))
else:
    print('Skipping - no images found')

In [ ]:
if 'df_quality' in dir() and not df_quality.empty:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('OPG Image Quality Analysis', fontsize=14, fontweight='bold')

    # Resolution distribution
    axes[0,0].scatter(df_quality['width'], df_quality['height'], alpha=0.6, s=50, c='#2196F3')
    axes[0,0].set_xlabel('Width (px)')
    axes[0,0].set_ylabel('Height (px)')
    axes[0,0].set_title('Image Resolutions')

    # Brightness distribution
    axes[0,1].hist(df_quality['mean_brightness'], bins=20, color='#4CAF50', alpha=0.8, edgecolor='white')
    axes[0,1].axvline(df_quality['mean_brightness'].mean(), color='red', linestyle='--', label='Mean')
    axes[0,1].set_xlabel('Mean Brightness')
    axes[0,1].set_title('Brightness Distribution')
    axes[0,1].legend()

    # Sharpness distribution
    axes[0,2].hist(df_quality['sharpness'], bins=20, color='#FF9800', alpha=0.8, edgecolor='white')
    axes[0,2].set_xlabel('Laplacian Variance (Sharpness)')
    axes[0,2].set_title('Sharpness Distribution')

    # Contrast distribution
    axes[1,0].hist(df_quality['contrast'], bins=20, color='#9C27B0', alpha=0.8, edgecolor='white')
    axes[1,0].set_xlabel('Contrast Score')
    axes[1,0].set_title('Contrast Distribution')

    # Megapixels distribution
    axes[1,1].hist(df_quality['megapixels'], bins=15, color='#F44336', alpha=0.8, edgecolor='white')
    axes[1,1].set_xlabel('Megapixels')
    axes[1,1].set_title('Image Size Distribution')

    # SNR distribution
    axes[1,2].hist(df_quality['snr'], bins=20, color='#00BCD4', alpha=0.8, edgecolor='white')
    axes[1,2].set_xlabel('Signal-to-Noise Ratio')
    axes[1,2].set_title('SNR Distribution')

    plt.tight_layout()
    plt.savefig('reports/figures/image_quality_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved: reports/figures/image_quality_analysis.png')

## 3. Annotation Analysis

In [ ]:
def analyze_yolo_annotations(label_paths: list) -> pd.DataFrame:
    """Analyze YOLO format annotation statistics."""
    records = []
    for lbl_path in label_paths:
        try:
            with open(lbl_path) as f:
                lines = [l.strip() for l in f if l.strip()]
            
            for line in lines:
                parts = line.split()
                if len(parts) == 5:
                    cls, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    area = bw * bh
                    records.append({
                        'class_id': cls,
                        'cx': cx, 'cy': cy,
                        'bw': bw, 'bh': bh,
                        'area': area,
                        'aspect_ratio': bw / bh if bh > 0 else 0,
                    })
        except Exception:
            pass
    
    return pd.DataFrame(records)


# Analyze processed dataset if available
if PROCESSED_DATA.exists():
    train_labels = list((PROCESSED_DATA / 'train' / 'labels').rglob('*.txt'))
    val_labels = list((PROCESSED_DATA / 'val' / 'labels').rglob('*.txt'))
    test_labels = list((PROCESSED_DATA / 'test' / 'labels').rglob('*.txt'))
    
    print(f'Processed dataset split:')
    print(f'  Train: {len(train_labels)} label files')
    print(f'  Val: {len(val_labels)} label files')
    print(f'  Test: {len(test_labels)} label files')
    
    all_labels_processed = train_labels + val_labels + test_labels
    if all_labels_processed:
        df_ann = analyze_yolo_annotations(all_labels_processed)
        print(f'\n  Total annotations: {len(df_ann)}')
        if not df_ann.empty:
            print(f'\n📦 Bounding Box Statistics:')
            display(df_ann[['bw', 'bh', 'area', 'aspect_ratio']].describe().round(4))
elif data_root:
    raw_labels = [f for f in data_root.rglob('*.txt') if f.suffix == '.txt' and 'classes' not in f.name.lower()]
    if raw_labels:
        df_ann = analyze_yolo_annotations(raw_labels)
        print(f'Raw annotations: {len(df_ann)}')
    else:
        print('No YOLO annotations found (may be COCO/VOC format)')

In [ ]:
if 'df_ann' in dir() and not df_ann.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Annotation Analysis — Cavity Detection', fontsize=14, fontweight='bold')

    # Bounding box center distribution (heatmap)
    h, xedges, yedges = np.histogram2d(df_ann['cx'], df_ann['cy'], bins=20)
    im = axes[0,0].imshow(h.T, origin='lower', extent=[0, 1, 0, 1],
                           aspect='auto', cmap='YlOrRd')
    plt.colorbar(im, ax=axes[0,0], label='Count')
    axes[0,0].set_xlabel('Normalized X (center)')
    axes[0,0].set_ylabel('Normalized Y (center)')
    axes[0,0].set_title('Cavity Location Heatmap')
    axes[0,0].invert_yaxis()

    # Bounding box size distribution
    axes[0,1].scatter(df_ann['bw'], df_ann['bh'], alpha=0.4, s=20, c='#2196F3')
    axes[0,1].set_xlabel('Normalized Width')
    axes[0,1].set_ylabel('Normalized Height')
    axes[0,1].set_title('Cavity Size Distribution')

    # Area distribution
    axes[1,0].hist(df_ann['area'] * 100, bins=30, color='#4CAF50', alpha=0.8, edgecolor='white')
    axes[1,0].set_xlabel('Normalized Area (%)')
    axes[1,0].set_title('Cavity Area Distribution')
    axes[1,0].axvline(df_ann['area'].mean() * 100, color='red', linestyle='--',
                      label=f'Mean: {df_ann["area"].mean()*100:.2f}%')
    axes[1,0].legend()

    # Annotations per image
    if PROCESSED_DATA.exists():
        ann_counts = []
        for lbl in (PROCESSED_DATA / 'train' / 'labels').rglob('*.txt'):
            with open(lbl) as f:
                count = sum(1 for line in f if line.strip())
            ann_counts.append(count)
        
        counter = Counter(ann_counts)
        axes[1,1].bar(counter.keys(), counter.values(), color='#FF9800', alpha=0.8, edgecolor='white')
        axes[1,1].set_xlabel('Cavities per Image')
        axes[1,1].set_ylabel('Number of Images')
        axes[1,1].set_title('Cavities per Image Distribution')

    plt.tight_layout()
    plt.savefig('reports/figures/annotation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved: reports/figures/annotation_analysis.png')

## 4. Sample Images with Annotations

In [ ]:
def visualize_sample_with_annotations(data_dir: Path, split: str = 'train', n_samples: int = 6):
    """Visualize sample OPG images with bounding box annotations."""
    img_dir = data_dir / split / 'images'
    lbl_dir = data_dir / split / 'labels'
    
    if not img_dir.exists():
        print(f'Split {split} not found')
        return
    
    images = list(img_dir.glob('*.jpg'))
    if not images:
        images = list(img_dir.glob('*.png'))
    
    if not images:
        print('No images found')
        return
    
    # Select samples with annotations
    annotated = []
    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                lines = [l.strip() for l in f if l.strip()]
            if lines:
                annotated.append((img_path, lines))
    
    if not annotated:
        # Show unannotated
        annotated = [(img, []) for img in images[:n_samples]]
    
    sample = annotated[:n_samples]
    
    cols = 3
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
    fig.suptitle(f'Sample OPG Images — {split.capitalize()} Split', fontsize=14, fontweight='bold')
    
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (img_path, labels) in enumerate(sample):
        row, col = idx // cols, idx % cols
        ax = axes[row, col]
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        ax.imshow(img_rgb, cmap='gray' if img_rgb.mean() < 50 else None)
        
        # Draw bounding boxes
        for label in labels:
            parts = label.split()
            if len(parts) == 5:
                cls, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                          linewidth=2, edgecolor='red', facecolor='red', alpha=0.2)
                rect_border = patches.Rectangle((x1, y1), bw*w, bh*h,
                                                 linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(rect)
                ax.add_patch(rect_border)
                ax.text(x1, y1-3, f'cavity {cls}', color='red', fontsize=8,
                       bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        
        ax.set_title(f'{img_path.name}\n{len(labels)} annotation(s)', fontsize=9)
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(len(sample), rows * cols):
        axes[idx // cols, idx % cols].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'reports/figures/sample_{split}_images.png', dpi=150, bbox_inches='tight')
    plt.show()


# Visualize from processed dataset
os.makedirs('reports/figures', exist_ok=True)

if PROCESSED_DATA.exists():
    visualize_sample_with_annotations(PROCESSED_DATA, split='train')
elif data_root:
    print('Processed dataset not yet created. Run: python main.py --stages 1,2,3')

## 5. Augmentation Preview

In [ ]:
def preview_augmentations(image_path: Path, n_augmentations: int = 6):
    """Preview medical-safe augmentations applied to an OPG image."""
    img = cv2.imread(str(image_path))
    if img is None:
        print(f'Cannot read image: {image_path}')
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    augmentations = {
        'Original': img_rgb,
    }
    
    # 1. CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(gray)
    augmentations['CLAHE Enhancement'] = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2RGB)
    
    # 2. Brightness +20%
    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=0)
    augmentations['Brightness +20%'] = cv2.cvtColor(bright, cv2.COLOR_BGR2RGB)
    
    # 3. Brightness -20%
    dark = cv2.convertScaleAbs(img, alpha=0.8, beta=0)
    augmentations['Brightness -20%'] = cv2.cvtColor(dark, cv2.COLOR_BGR2RGB)
    
    # 4. Slight rotation (+3°)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), 3, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    augmentations['Rotation +3°'] = cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB)
    
    # 5. Gaussian blur
    blurred = cv2.GaussianBlur(img, (3, 3), 1.0)
    augmentations['Gaussian Blur'] = cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)
    
    # 6. Sharpening
    kernel = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]])
    sharpened = cv2.filter2D(img, -1, kernel)
    augmentations['Sharpening'] = cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)
    
    cols = 3
    rows = (len(augmentations) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    fig.suptitle('Medical-Safe Augmentation Preview for OPG X-rays', fontsize=14, fontweight='bold')
    
    for idx, (aug_name, aug_img) in enumerate(augmentations.items()):
        ax = axes[idx // cols, idx % cols] if rows > 1 else axes[idx % cols]
        ax.imshow(aug_img)
        ax.set_title(aug_name, fontweight='bold' if aug_name == 'Original' else 'normal')
        ax.axis('off')
    
    # Hide empty
    total = rows * cols
    for idx in range(len(augmentations), total):
        ax = axes[idx // cols, idx % cols] if rows > 1 else axes[idx % cols]
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('reports/figures/augmentation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()


# Run augmentation preview on first available image
if PROCESSED_DATA.exists():
    train_imgs = list((PROCESSED_DATA / 'train' / 'images').glob('*.jpg'))
    if train_imgs:
        preview_augmentations(train_imgs[0])
elif data_root and 'all_images' in dir() and all_images:
    preview_augmentations(all_images[0])

## 6. Training Results Analysis (Post-Training)

In [ ]:
results_csv = Path('artifacts/model_trainer/results/cavity_detection/results.csv')

if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()
    print('Training results loaded:')
    display(df_results.tail(5))
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('YOLOv8 Training Progress', fontsize=14, fontweight='bold')
    
    epoch = range(1, len(df_results) + 1)
    
    # Losses
    for col, ax, color, title in [
        ('train/box_loss', axes[0,0], '#F44336', 'Box Loss (Train)'),
        ('train/cls_loss', axes[0,1], '#FF9800', 'Class Loss (Train)'),
        ('val/box_loss', axes[0,2], '#2196F3', 'Box Loss (Val)'),
    ]:
        if col in df_results.columns:
            ax.plot(epoch, df_results[col], color=color, linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(title)
            ax.grid(True, alpha=0.3)
    
    # Metrics
    for col, ax, color, title in [
        ('metrics/mAP50(B)', axes[1,0], '#4CAF50', 'mAP@0.5'),
        ('metrics/precision(B)', axes[1,1], '#9C27B0', 'Precision'),
        ('metrics/recall(B)', axes[1,2], '#00BCD4', 'Recall'),
    ]:
        if col in df_results.columns:
            ax.plot(epoch, df_results[col], color=color, linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Score')
            ax.set_title(title)
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            
            # Mark best epoch
            best_epoch = df_results[col].idxmax() + 1
            best_val = df_results[col].max()
            ax.scatter([best_epoch], [best_val], color='red', s=100, zorder=5,
                      label=f'Best: {best_val:.3f} @ epoch {best_epoch}')
            ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.savefig('reports/figures/training_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved: reports/figures/training_progress.png')
else:
    print('Training results not yet available. Run: python main.py --stage 4')

## 7. Summary

In [ ]:
print('=' * 60)
print('EDA SUMMARY — DENTAL OPG CAVITY DETECTION')
print('=' * 60)

if data_root and 'all_images' in dir():
    print(f'Total images: {len(all_images)}')

if PROCESSED_DATA.exists():
    for split in ['train', 'val', 'test']:
        split_img_dir = PROCESSED_DATA / split / 'images'
        if split_img_dir.exists():
            n = len(list(split_img_dir.glob('*.jpg')))
            print(f'{split.capitalize()} images: {n}')

if 'df_ann' in dir() and not df_ann.empty:
    print(f'Total annotations: {len(df_ann)}')
    print(f'Avg annotation area: {df_ann["area"].mean()*100:.2f}%')

print('\nNext steps:')
print('  1. python main.py --stage 4  # Train model')
print('  2. python main.py --stage 5  # Evaluate model')
print('  3. python app/gradio_app.py  # Launch web app')
print('  4. python scripts/deploy_to_huggingface.py --token HF_TOKEN')
print('=' * 60)